In [2]:
# ==========================================
# Fake News Detection using Traditional ML
# TF-IDF + Logistic Regression, SVM, Naive Bayes
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [3]:
# ==========================================
# 1. Load Dataset
# ==========================================

df = pd.read_csv(r"C:\Amit Dubli\Kaggle\Fake news\Real_and_Fake_News_Dataset.csv")

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (45757, 2)
                                                text  label
0  Gere faults Trump for blurring meaning of 'ref...      1
1  German parties start to find common ground in ...      1
2  Senate Democratic leader says Attorney General...      1
3  Tennis: Kyrgios fined $10,000 for Shanghai wal...      1
4   Trump Threw Mar-A-Lago Fundraiser For Woman A...      0


In [4]:
# ==========================================
# 2. Basic Data Cleaning
# ==========================================

df = df[['text', 'label']]

# Remove missing values
df.dropna(inplace=True)

# Remove duplicate articles
df.drop_duplicates(subset='text', inplace=True)

print("\nDataset Shape After Cleaning:", df.shape)


Dataset Shape After Cleaning: (45757, 2)


In [5]:
# ==========================================
# 3. Features and Target
# ==========================================

X = df['text']
y = df['label']

In [6]:
# ==========================================
# 4. Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Training Samples: 36605
Testing Samples: 9152


In [7]:
# ==========================================
# 5. TF-IDF Vectorization
# ==========================================

tfidf = TfidfVectorizer(
    stop_words='english',
    max_df=0.7,
    min_df=2,
    ngram_range=(1,2),
    max_features=30000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nTF-IDF Matrix Shape:", X_train_tfidf.shape)


TF-IDF Matrix Shape: (36605, 30000)


In [8]:
# ==========================================
# 6. Models
# ==========================================

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Linear SVM": LinearSVC(
        random_state=42
    ),

    "Multinomial Naive Bayes": MultinomialNB()
}

In [9]:
# ==========================================
# 7. Training & Evaluation
# ==========================================

results = []

for model_name, model in models.items():

    print("\n" + "="*60)
    print(f"Training: {model_name}")
    print("="*60)

    # Train
    model.fit(X_train_tfidf, y_train)

    # Predict
    y_pred = model.predict(X_test_tfidf)

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy
    })

    print(f"\nAccuracy: {accuracy:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Training: Logistic Regression

Accuracy: 0.9772

Confusion Matrix:
[[4475   97]
 [ 112 4468]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      4572
           1       0.98      0.98      0.98      4580

    accuracy                           0.98      9152
   macro avg       0.98      0.98      0.98      9152
weighted avg       0.98      0.98      0.98      9152


Training: Linear SVM

Accuracy: 0.9858

Confusion Matrix:
[[4523   49]
 [  81 4499]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      4572
           1       0.99      0.98      0.99      4580

    accuracy                           0.99      9152
   macro avg       0.99      0.99      0.99      9152
weighted avg       0.99      0.99      0.99      9152


Training: Multinomial Naive Bayes

Accuracy: 0.9339

Confusion Matrix:
[[4311  261]
 [ 344 4236]]

Classification Re

In [10]:
# ==========================================
# 8. Compare Results
# ==========================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n")
print("="*60)
print("MODEL COMPARISON")
print("="*60)
print(results_df)



MODEL COMPARISON
                     Model  Accuracy
1               Linear SVM  0.985795
0      Logistic Regression  0.977163
2  Multinomial Naive Bayes  0.933894


In [19]:
# ==========================================
# 9. Predict New News Article
# ==========================================

best_model = LinearSVC(
   # max_iter=1000,
    random_state=42
)

best_model.fit(X_train_tfidf, y_train)

sample_news = """
Aliens secretly control world governments
"""

sample_vector = tfidf.transform([sample_news])

prediction = best_model.predict(sample_vector)[0]

if prediction == 0:
    print("\nPrediction: FAKE NEWS")
else:
    print("\nPrediction: REAL NEWS")


Prediction: FAKE NEWS
